# Lab - Confidence Calibration and Human Review

Three tasks on spending the reviewers' time where the errors really are - and deciding
that with evidence, not with the model's own word. ShopAssist extracts two thousand
documents a week; the reviewers can check two hundred.

| Task | What you learn |
| --- | --- |
| 1 | Field-level confidence is useful only after it is calibrated on known answers |
| 2 | An overall accuracy is an average: check every document type and field first |
| 3 | Keep sampling every group - stratified - including the high-confidence ones |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Python and Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import random

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# How accurate a score bucket or a group must be before we trust it without review.
TARGET = 0.95

# The labeled validation set: 500 past documents - three return forms in four, one
# email in five, one invoice in twenty - where a person wrote down the correct value
# of every field. ShopAssist ran on all of them; code compared each field with its
# label. Per (document type, field): (score Claude gave, fields, how many were right).
LABELED = {
    ("return_form", "order_id"): [(10, 300, 300), (9, 60, 60), (8, 10, 10), (7, 5, 5)],
    ("return_form", "reason"): [(10, 200, 199), (9, 120, 119), (8, 30, 28), (7, 20, 17),
                                (6, 5, 3)],
    ("email", "order_id"): [(10, 70, 70), (9, 15, 15), (8, 5, 5), (7, 4, 4), (6, 4, 2),
                            (5, 2, 1)],
    ("email", "reason"): [(10, 30, 30), (9, 30, 30), (8, 20, 19), (7, 15, 12), (6, 5, 3)],
    ("invoice", "order_id"): [(10, 20, 20), (9, 5, 4)],
    ("invoice", "refund_amount"): [(10, 12, 8), (9, 8, 5), (8, 5, 3)],
}
# One row per extracted field: its document type, field, score, and whether it was right.
VALIDATION = [
    {"doc_type": doc_type, "field": field, "score": score, "correct": i < right}
    for (doc_type, field), buckets in LABELED.items()
    for score, count, right in buckets
    for i in range(count)
]

# This week's output, already split by document type and confidence level. A few
# items in every group are wrong; nobody knows which until a reviewer looks.
WEEK = {   # (doc_type, level): (items, wrong, what a reviewer finds)
    ("return_form", "high"): (1200, 6, "order number mistyped"),
    ("return_form", "low"): (300, 15, "return reason misread"),
    ("email", "high"): (320, 5, "order number mistyped"),
    ("email", "low"): (80, 8, "return reason misread"),
    ("invoice", "high"): (80, 16, "date with day and month swapped"),
    ("invoice", "low"): (20, 4, "refund amount misread"),
}
THIS_WEEK = []
for (doc_type, level), (count, wrong, problem) in WEEK.items():
    for i in range(count):
        THIS_WEEK.append({"id": "DOC-{:04d}".format(len(THIS_WEEK) + 1),
                          "doc_type": doc_type, "level": level,
                          "problem": problem if i % (count // wrong) == 0 else None})


def review(item):
    """A reviewer opens the extraction next to its source and reports what is wrong."""
    return item["problem"]

---

## Task 1 - Calibrate field-level confidence

In [ ]:
# ============================================================
# TASK 1 - Calibrate field-level confidence
# ============================================================
#
# WHAT TO DO
#   First, see field-level confidence: Claude scores each field
#   it extracts (provided). Then finish calibrate(): group the
#   labeled results by the score Claude gave, walk down from
#   10, and stop at the first score whose accuracy falls below
#   TARGET. The last score that held is the field's cut-off.
#
# WHY IT MATTERS
#   A score of nine out of ten sounds like evidence. On its own
#   it is not: Claude produces the score the same way it
#   produces the value, so it can be confidently wrong. A score
#   becomes useful only after you check it against known
#   answers. Each field gets its own cut-off - the order number
#   is printed clearly, the return reason is a vague sentence.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Start with confidence".
#
# HOW TO DO IT
#   Three blanks:
#
#       score    keep the rows Claude gave exactly this score
#       TARGET   the accuracy a score must reach
#       score    the lowest score that held so far
# ============================================================

# provided: one extraction, with a confidence score next to each field
SCORE = {"type": "integer", "minimum": 1, "maximum": 10}
SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": ["string", "null"]},
        "order_id_confidence": SCORE,
        "reason": {"type": "string", "enum": ["too_small", "too_big", "damaged", "other"]},
        "reason_confidence": SCORE,
    },
    "required": ["order_id", "order_id_confidence", "reason", "reason_confidence"],
    "additionalProperties": False,
}
EMAIL = ("Hi, my order ORD-12345678 came yesterday and honestly it is just not "
         "what I hoped for. I would like to send it back. Sam")
response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    output_config={"effort": "low",
                   "format": {"type": "json_schema", "schema": SCHEMA}},
    messages=[{"role": "user", "content": "Extract the return request from this "
               "email. Give each field a confidence score from 1 to 10.\n\n" + EMAIL}],
)
extraction = json.loads(response.content[0].text)
print("Claude says:", extraction)


def accuracy(rows):
    return sum(1 for r in rows if r["correct"]) / len(rows)


# BEGIN SOLUTION
def calibrate(field):
    """The lowest score from which every score bucket reaches TARGET."""
    cutoff = None
    for score in range(10, 0, -1):
        rows = [r for r in VALIDATION if r["field"] == field and r["score"] == score]
        if not rows:
            continue
        if accuracy(rows) < TARGET:
            break
        cutoff = score
    return cutoff
# SCAFFOLD: def calibrate(field):
# SCAFFOLD:     """The lowest score from which every score bucket reaches TARGET."""
# SCAFFOLD:     cutoff = None
# SCAFFOLD:     for score in range(10, 0, -1):
# SCAFFOLD:         rows = [r for r in VALIDATION if r["field"] == field and r["score"] == ...]   # score
# SCAFFOLD:         if not rows:
# SCAFFOLD:             continue
# SCAFFOLD:         if accuracy(rows) < ...:   # TARGET
# SCAFFOLD:             break
# SCAFFOLD:         cutoff = ...   # score - this one held
# SCAFFOLD:     return cutoff
# END SOLUTION: replace each ... below with the value named beside it

for score in (10, 9, 8, 7):   # provided: the reason field, grouped by score
    rows = [r for r in VALIDATION if r["field"] == "reason" and r["score"] == score]
    print("reason scored", score, "->", "{:.0%}".format(accuracy(rows)), "right")

CUTOFFS = {field: calibrate(field) for field in ("order_id", "reason")}
print("cut-offs:", CUTOFFS)
if not all(isinstance(c, int) for c in CUTOFFS.values()):   # provided: said plainly
    raise ValueError("calibrate() found no cut-off. Replace each ... in it with the "
                     "value named beside it.")

try:   # provided: route this extraction - a field below its cut-off goes to a reviewer
    below = [f for f in CUTOFFS if extraction[f + "_confidence"] < CUTOFFS[f]]
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("goes to a reviewer:" if below else "goes straight through", below)

check("calibrate", extraction=extraction, cutoffs=CUTOFFS)

---

## Task 2 - Accuracy per document type and field

In [ ]:
# ============================================================
# TASK 2 - Accuracy per document type and field
# ============================================================
#
# WHAT TO DO
#   The team lead sees 97% accuracy overall and proposes to stop
#   reviewing high-confidence extractions. Break the same
#   results down by document type, then the invoices by field,
#   and automate only the types that reach TARGET.
#
# WHY IT MATTERS
#   An overall number is an average, and an average can hide a
#   weak part. Return forms are three documents in four, and
#   they pull the average up. Invoices are one in twenty - and
#   on invoices, the field that is wrong most often is the one
#   that sends money to a customer.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Break the same results down by document type".
#
# HOW TO DO IT
#   Three blanks:
#
#       t        keep the rows of this document type
#       f        keep the invoice rows of this field
#       TARGET   what a type must reach to be automated
# ============================================================

TYPES = ("return_form", "email", "invoice")
print("overall: {:.1%}".format(accuracy(VALIDATION)))

try:
    # BEGIN SOLUTION
    by_type = {t: accuracy([r for r in VALIDATION if r["doc_type"] == t])
               for t in TYPES}
    invoice_fields = {
        f: accuracy([r for r in VALIDATION
                     if r["doc_type"] == "invoice" and r["field"] == f])
        for f in ("order_id", "refund_amount")
    }
    AUTOMATE = [t for t in TYPES if by_type[t] >= TARGET]
    # SCAFFOLD: by_type = {t: accuracy([r for r in VALIDATION if r["doc_type"] == ...])   # t
    # SCAFFOLD:            for t in TYPES}
    # SCAFFOLD: invoice_fields = {
    # SCAFFOLD:     f: accuracy([r for r in VALIDATION
    # SCAFFOLD:                  if r["doc_type"] == "invoice" and r["field"] == ...])   # f
    # SCAFFOLD:     for f in ("order_id", "refund_amount")
    # SCAFFOLD: }
    # SCAFFOLD: AUTOMATE = [t for t in TYPES if by_type[t] >= ...]   # TARGET
    # END SOLUTION: replace each ... below with the value named beside it
except ZeroDivisionError:   # provided: a filter that kept no rows, said plainly
    raise ValueError("A filter kept no rows. Replace each ... above with the value "
                     "named beside it.") from None
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

for t in TYPES:
    share = sum(1 for r in VALIDATION if r["doc_type"] == t) / len(VALIDATION)
    print("  {:12} {:4.0%} of documents, {:.1%} right".format(t, share, by_type[t]))
print("invoices by field:", {f: "{:.0%}".format(a) for f, a in invoice_fields.items()})
print("automate:", AUTOMATE)

check("per_group", by_type=by_type, invoice_fields=invoice_fields, automate=AUTOMATE)

---

## Task 3 - Keep sampling every group

In [ ]:
# ============================================================
# TASK 3 - Keep sampling every group
# ============================================================
#
# WHAT TO DO
#   Finish stratified_sample(): split this week's output into
#   groups by document type and confidence level, then pick a
#   few items at random from every group. The reviewers check
#   twenty from each group.
#
# WHY IT MATTERS
#   Once a group skips review, nobody looks at it - and errors
#   can appear exactly where nothing looks uncertain. Plain
#   random picks are mostly return forms; invoices hardly
#   appear. Sampling every group measures the real error rate
#   of each one, and finds patterns the validation set never
#   contained.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So you first split the output into groups".
#
# HOW TO DO IT
#   Three blanks:
#
#       "level"     the second key of a group: high or low confidence
#       members     the items of one group, to pick from
#       per_group   how many to pick, at most
# ============================================================

# BEGIN SOLUTION
def stratified_sample(items, per_group, rng):
    groups = {}
    for item in items:
        key = (item["doc_type"], item["level"])
        groups.setdefault(key, []).append(item)
    sample = []
    for key, members in sorted(groups.items()):
        sample += rng.sample(members, min(per_group, len(members)))
    return sample
# SCAFFOLD: def stratified_sample(items, per_group, rng):
# SCAFFOLD:     groups = {}
# SCAFFOLD:     for item in items:
# SCAFFOLD:         key = (item["doc_type"], item[...])   # "level" - high or low confidence
# SCAFFOLD:         groups.setdefault(key, []).append(item)
# SCAFFOLD:     sample = []
# SCAFFOLD:     for key, members in sorted(groups.items()):
# SCAFFOLD:         sample += rng.sample(..., min(..., len(members)))   # members, per_group
# SCAFFOLD:     return sample
# END SOLUTION: replace each ... below with the value named beside it

try:
    sample = stratified_sample(THIS_WEEK, 20, random.Random(2024))
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
plain = random.Random(2024).sample(THIS_WEEK, len(sample))   # provided: for comparison
print("plain random picks:", sum(1 for i in plain if i["doc_type"] == "invoice"),
      "invoices of", len(plain))

found = {}   # provided: the reviewers check every item in the sample
for item in sample:
    key = (item["doc_type"], item["level"])
    if review(item):
        found[key] = found.get(key, 0) + 1
        print("  wrong:", key, "-", review(item))
for key in sorted({(i["doc_type"], i["level"]) for i in THIS_WEEK}):
    checked = sum(1 for i in sample if (i["doc_type"], i["level"]) == key)
    print("  {:28} checked {:2}, wrong {}".format(str(key), checked, found.get(key, 0)))

check("stratified", sample=sample, items=THIS_WEEK, found=found)

---

## Done

Decide human review with evidence, not with the model's own word. Calibrate field-level
confidence on a labeled validation set, and send low-confidence items to people first.
Check accuracy for every document type and every field before you automate any of them.
And keep sampling every group, including the high-confidence one: the swapped dates
scored high, so no cut-off could catch them - the sample did. Those invoices go into the
labeled set, the cut-offs are calibrated again, and the group stays under review until
it holds up.